# Penguins Dataset - Logistic Regression

### [Penguins Dataset](https://seaborn.pydata.org/tutorial/introduction.html)

Author: [Kevin Thomas](mailto:ket189@pitt.edu)

## Citation

[1] Allison Horst, https://github.com/allisonhorst/palmerpenguins

## Import Modules

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from patsy import dmatrices
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

## Load Dataset

In [ ]:
df = pd.read_csv('penguins-clean.csv')

In [ ]:
df.head()

## Logistic Regression

Logistic regression predicts the probability that a sample belongs to a class. We turn the three-class `species` target into a binary target, `is_gentoo`, and model the probability that a penguin is a Gentoo.

### Create a Binary Target

In [ ]:
df['is_gentoo'] = (df['species'] == 'Gentoo').astype(int)
df['is_gentoo'].value_counts()

The classes are reasonably balanced, so accuracy is a fair starting metric, but we will still inspect ROC and AUC.

### Visualize Relationship Between the Target and Each Continuous Input

Gentoo penguins have longer flippers and deeper bills. `flipper_length_mm` separates the classes most cleanly.

In [ ]:
sns.pairplot(
    df,
    vars=['bill_length_mm',
          'bill_depth_mm',
          'flipper_length_mm'],
    hue='is_gentoo')
plt.show()

## Fit Logistic Regression Models - Training Data

### Functions

`fit_and_assess_logit` fits a model with the statsmodels formula API and returns one row of metrics. `predict_class` turns probabilities into class labels at a 0.5 threshold.

In [ ]:
def fit_and_assess_logit(model, formula, df):
    """
    Fit a logistic regression model and return its metrics.

    Parameters:
        model (str): A descriptive name for the model.
        formula (str): A statsmodels formula string.
        df (pandas.DataFrame): The input data.

    Returns:
        pandas.DataFrame: A single-row DataFrame of metrics.
    """
    a_model = smf.logit(formula=formula, data=df).fit(disp=0)
    preds = predict_class(a_model, df)
    results_dict = {'model_name': model,
                    'model_formula': formula,
                    'num_coefs': len(a_model.params),
                    'pseudo_r2': a_model.prsquared,
                    'aic': a_model.aic,
                    'accuracy': (preds == df['is_gentoo']).mean()}
    return pd.DataFrame(results_dict, index=[0])


def predict_class(a_model, df):
    """
    Predict binary class labels from a fitted model.

    Parameters:
        a_model (statsmodels.discrete.discrete_model.BinaryResultsWrapper): Fitted model.
        df (pandas.DataFrame): The input data.

    Returns:
        numpy.ndarray: Predicted class labels.
    """
    probs = a_model.predict(df)
    return (probs >= 0.5).astype(int)

### Formulas

In [ ]:
formula_00 = 'is_gentoo ~ 1'
formula_01 = 'is_gentoo ~ flipper_length_mm'
formula_02 = 'is_gentoo ~ bill_length_mm + bill_depth_mm + flipper_length_mm'
formula_03 = 'is_gentoo ~ bill_length_mm + bill_depth_mm + flipper_length_mm + C(island)'
formula_04 = 'is_gentoo ~ bill_length_mm + bill_depth_mm + flipper_length_mm + C(island) + C(sex)'
formula_05 = 'is_gentoo ~ flipper_length_mm * C(island)'
formula_05

### Model 00: Intercept-Only (Baseline)

In [ ]:
fit_and_assess_logit('Model 00', formula_00, df)

### Model 01: Flipper Length Only

In [ ]:
fit_and_assess_logit('Model 01', formula_01, df)

### Model 02: All Continuous Predictors

In [ ]:
fit_and_assess_logit('Model 02', formula_02, df)

### Model 03: Continuous Predictors + Island

In [ ]:
fit_and_assess_logit('Model 03', formula_03, df)

### Model 04: Continuous Predictors + Island + Sex

In [ ]:
fit_and_assess_logit('Model 04', formula_04, df)

### Model 05: Flipper Length Interacting With Island

In [ ]:
fit_and_assess_logit('Model 05', formula_05, df)

### Compare Fitted Models - Training Data

Model 03 and Model 04 reach the highest accuracy with the fewest coefficients. Higher pseudo R-squared and lower AIC both indicate a better fit.

In [ ]:
training_results = pd.concat([
    fit_and_assess_logit('Model 00', formula_00, df),
    fit_and_assess_logit('Model 01', formula_01, df),
    fit_and_assess_logit('Model 02', formula_02, df),
    fit_and_assess_logit('Model 03', formula_03, df),
    fit_and_assess_logit('Model 04', formula_04, df),
    fit_and_assess_logit('Model 05', formula_05, df)])
training_results.sort_values('accuracy', ascending=False)

## Fit Logistic Regression Models w/ Cross-Validation - Test Data

### Functions

In [ ]:
def logit_cross_val_score(model, formula, init_model, df, cv):
    """
    Run cross-validated evaluation for a logistic regression formula.

    Parameters:
        model (str): A human-readable model name.
        formula (str): A Patsy-compatible formula string.
        init_model (object): An unfitted scikit-learn estimator.
        df (pandas.DataFrame): The input data.
        cv (int): Number of cross-validation folds.

    Returns:
        pandas.DataFrame: Per-fold accuracy scores.
    """
    y, X = dmatrices(formula, data=df, return_type='dataframe')
    y = np.ravel(y)
    scores = cross_val_score(init_model, X, y, cv=cv, scoring='accuracy')
    return pd.DataFrame({'model_name': model,
                         'fold': range(1, cv + 1),
                         'accuracy': scores})

### Fit w/ Cross-Validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_models = [('Model 00', formula_00),
             ('Model 01', formula_01),
             ('Model 02', formula_02),
             ('Model 03', formula_03),
             ('Model 04', formula_04),
             ('Model 05', formula_05)]
cv_results = pd.concat([
    logit_cross_val_score(name, formula, LogisticRegression(max_iter=5000), df, 5)
    for name, formula in cv_models])
cv_results.head()

### Compare Fitted Models - Test Data

The cross-validated accuracy is the honest estimate. Model 03 is a strong choice: high accuracy with fewer coefficients than Model 04.

In [ ]:
cv_summary = cv_results.groupby('model_name')['accuracy'].agg(['mean', 'std', 'count'])
cv_summary = cv_summary.sort_values('mean', ascending=False)
cv_summary

### Visualize Average Accuracy w/ 1 Standard Error Interval

In [ ]:
ax = cv_summary['mean'].plot(
    kind='bar',
    yerr=cv_summary['std'] / np.sqrt(cv_summary['count']),
    capsize=4,
    figsize=(10, 6))
ax.set_ylabel('Mean accuracy')
ax.set_title('Cross-validated accuracy by model')
plt.tight_layout()
plt.show()

### Visualize ROC Curves

A model with a high AUC ranks Gentoo above non-Gentoo across all thresholds. Model 03 is close to a perfect ranker.

In [ ]:
y, X = dmatrices(formula_03, data=df, return_type='dataframe')
y = np.ravel(y)
best_model = LogisticRegression(max_iter=5000).fit(X, y)
probs = best_model.predict_proba(X)[:, 1]
fpr, tpr, _ = roc_curve(y, probs)
auc_score = roc_auc_score(y, probs)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Model 03 (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve - Model 03')
plt.legend()
plt.tight_layout()
plt.show()

### Confusion Matrix and Classification Report

In [ ]:
preds = (probs >= 0.5).astype(int)
print(confusion_matrix(y, preds))
print(classification_report(y, preds))

## Save Best Model - Model 03 w/ Highest Cross-Validated Accuracy

### Save Model

In [ ]:
final_model = smf.logit(formula=formula_03, data=df).fit(disp=0)
with open('logit_model.pkl', 'wb') as handle:
    pickle.dump(final_model, handle)
print(final_model.summary())

### Load Model

In [ ]:
with open('logit_model.pkl', 'rb') as handle:
    loaded_model = pickle.load(handle)
loaded_model.params

### Inference on New Data

Feed the loaded model a new penguin and read the probability that it is a Gentoo.

In [ ]:
new_penguin = pd.DataFrame({
    'bill_length_mm': [47.5],
    'bill_depth_mm': [15.0],
    'flipper_length_mm': [217.0],
    'island': ['Biscoe'],
    'sex': ['Female']})
probability = loaded_model.predict(new_penguin).iloc[0]
print(f'Probability of Gentoo: {probability:.4f}')
print('Predicted class:', 'Gentoo' if probability >= 0.5 else 'Not Gentoo')